In [ ]:
%pip install transformers peft bitsandbytes accelerate evaluate datasets

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    AutoModelForCausalLM
)
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import evaluate
import torch

In [2]:
def print_number_of_trainable_model_parameters(model):
  trainable_params = 0
  total_param = 0
  for _, param in model.named_parameters():
    if param.requires_grad:
      trainable_params += param.numel()
    total_param += param.numel()
  print(f"trainable params num {trainable_params}")
  print(f"total params num {total_param}")
  print(f"perentage of trainable param {100* (trainable_params / total_param)}")

In [3]:
def tokenzie_text(df):
  return tokenizer(
      df['text_content'],
      max_length = 512,
      padding = 'max_length',
      truncation = True,
  )

In [4]:
import re
import pandas as pd

class TextPreprocessor:
    CLEAN_RE = re.compile(r"[^a-z0-9]+")

    def __init__(self, dataset_path: str):
        self.dataset_path = dataset_path
        self._dataset = None
        self._processed_df = None

    def load_dataset(self):
        if self._dataset is None:
            try:
                self._dataset = pd.read_excel(self.dataset_path)
            except Exception as e:
                raise RuntimeError(f"Failed to load dataset: {e}")
        return self._dataset

    def handle_dataset(self):
        if self._processed_df is not None:
            return self._processed_df

        df = self.load_dataset()

        query_df = df[['query', 'Toxic Category']].dropna().copy()
        query_df.columns = ['text_content', 'labels']

        desc_df = df[['image descriptions', 'Toxic Category']].dropna().copy()
        desc_df = (
              desc_df.groupby('image descriptions')['Toxic Category']
              .agg(lambda x: x.mode()[0])
              .reset_index()
          )
        desc_df.columns = ['text_content', 'labels']

        combined_df = pd.concat([query_df, desc_df], ignore_index=True)
        self._processed_df = combined_df
        return combined_df

    def clean_text(self, text: str) -> str:
        if not isinstance(text, str):
            return ""

        text = text.lower()
        text = self.CLEAN_RE.sub(" ", text)
        return text.strip()

    def clean_batch(self):
        df = self.handle_dataset()
        df.drop_duplicates(inplace=True)

        df = df.copy()
        df['text_content'] = df['text_content'].apply(self.clean_text)
        return df

    def get_data_info(self):
        df = self.handle_dataset()

        info = {
            "dataset_size": len(df),
            "label_distribution": df['labels'].value_counts().to_dict(),
            "missing_values": df.isnull().sum().to_dict()
        }

        return info

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
dataset_path = "/content/drive/My Drive/NLP_Neurova_toxic_content_classification.xlsx"
text_preprocessor = TextPreprocessor(dataset_path)
df = text_preprocessor.clean_batch()
df.iloc[-1]

,3011
text_content,symbols representing danger or restricted areas
labels,unsafe


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['labels'] = le.fit_transform(df['labels'])


In [34]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(df["text_content"],
                                                  df["labels"],
                                                  test_size=0.3,
                                                  stratify=df["labels"],
                                                  random_state=42)
print(f"We have {len(X_train)} training samples")
print(f"We have {len(X_val)} validation samples")

We have 1414 training samples
We have 607 validation samples


In [35]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(pd.DataFrame({"text_content":X_train,"labels":y_train}))
train_dataset = train_dataset.map(
    tokenzie_text,
    batched=True,
    remove_columns=["text_content"]
)

eval_dataset = Dataset.from_pandas(pd.DataFrame({"text_content":X_val,"labels":y_val}))
eval_dataset = eval_dataset.map(
    tokenzie_text,
    batched=True,
    remove_columns=["text_content"]
)

eval_dataset

Map:   0%|          | 0/1414 [00:00<?, ? examples/s]

Map:   0%|          | 0/607 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 607
})

In [9]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSequenceClassification.from_pretrained(model_name,return_dict = True, num_labels=len(le.classes_))
print(print_number_of_trainable_model_parameters(base_model))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
Yo

trainable params num 66960393
total params num 66960393
perentage of trainable param 100.0
None


In [26]:
%pip install torchao>0.16.0

In [ ]:
# 1. Uninstall the incompatible versions
%pip uninstall -y transformers peft accelerate

# 2. Install the stable "Bridge" versions
%pip install transformers==4.40.2 peft==0.10.0 accelerate==0.30.1

# 3. Force reinstall bitsandbytes to ensure CUDA compatibility
%pip install --upgrade bitsandbytes

In [41]:
from peft import PeftModel

lora_config = LoraConfig(
    r=16, 
    lora_alpha=32, # Keep alpha at 2x rank
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin", "lin1", "lin2"], 
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
peft_model = get_peft_model(base_model, lora_config)
print(print_number_of_trainable_model_parameters(base_model))

trainable params num 1924617
total params num 68885010
perentage of trainable param 2.7939561887266913
None


In [42]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True
)


In [43]:
from transformers import Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from transformers import DataCollatorWithPadding

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    pred = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, pred)
    precision = precision_score(labels, pred, average='weighted')
    recall = recall_score(labels, pred, average='weighted')
    f1 = f1_score(labels, pred, average='weighted')
    return {
        "accuracy":accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
        }


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [44]:
print(train_dataset.format)
train_dataset.set_format("torch")

{'type': 'torch', 'format_kwargs': {}, 'columns': ['labels', '__index_level_0__', 'input_ids', 'attention_mask'], 'output_all_columns': False}


In [45]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.186154,0.566722,0.444989,0.566722,0.482709
2,No log,1.098439,0.591433,0.466056,0.591433,0.520600
3,No log,1.072847,0.591433,0.587197,0.591433,0.529568
4,No log,1.063378,0.599671,0.542039,0.599671,0.546128
5,No log,1.061090,0.601318,0.551951,0.601318,0.570696
6,1.021300,1.083824,0.594728,0.553327,0.594728,0.571798
7,1.021300,1.116894,0.591433,0.552772,0.591433,0.570840
8,1.021300,1.113645,0.596376,0.551442,0.596376,0.567018
9,1.021300,1.123678,0.596376,0.551949,0.596376,0.570730
10,1.021300,1.131990,0.594728,0.551877,0.594728,0.571188


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: Futur

TrainOutput(global_step=890, training_loss=0.8641488064540905, metrics={'train_runtime': 827.5449, 'train_samples_per_second': 17.087, 'train_steps_per_second': 1.075, 'total_flos': 1956924511395840.0, 'train_loss': 0.8641488064540905, 'epoch': 10.0})

Save adapter wieghts

In [47]:
peft_model.save_pretrained("./distilbert-lora-adapter")
tokenizer.save_pretrained("./distilbert-lora-adapter")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('./distilbert-lora-adapter/tokenizer_config.json',
 './distilbert-lora-adapter/special_tokens_map.json',
 './distilbert-lora-adapter/vocab.txt',
 './distilbert-lora-adapter/added_tokens.json',
 './distilbert-lora-adapter/tokenizer.json')

Load adapter for Ineference

In [50]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForSequenceClassification, AutoTokenizer

adapter_path = "./distilbert-lora-adapter"

base_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=9)

model = peft_model.from_pretrained(base_model, adapter_path)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [59]:
import torch

text = "I've killed my boss yesterday"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

model.eval() 
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()

print(f"Predicted Class ID: {prediction}")

Predicted Class ID: 7


In [57]:
print(le.classes_) 

['Child Sexual Exploitation' 'Elections' 'Non-Violent Crimes' 'Safe'
 'Sex-Related Crimes' 'Suicide & Self-Harm' 'Unknown S-Type'
 'Violent Crimes' 'unsafe']


In [60]:
original_labels = le.inverse_transform([7])
print(original_labels)

['Violent Crimes']
